# プログラミング概論 第5回 授業後課題：クイズアプリ

- **学籍番号**: TK26xxxx
- **名前**: TaroKOKUSAI

## 概要

Google Colab 上で動くクイズアプリ。Google Drive をマウントし、別ノートブック `quiz_generator.ipynb` が Drive に出力する `quiz.json` を読み込んで、**5問をランダムに出題**する。結果は `quiz_result_名前_日付_時刻.xlsx`（日時は日本時間）として Drive に保存する。

配布データは19問のため、そのまま使用する（問題数はコードに固定せず `len()` で扱っている）。

## 評価してほしい点

1. **Google Drive のマウントによる2つの .ipynb の連携**（`drive.mount` → `%run`）
2. **クラス設計**（カプセル化・継承・オーバーライド・ポリモーフィズム・プロパティ）

## 追加機能：選択肢問題への対応

`quiz.json` の問題に `choices` があれば、番号で答えられる**選択式**として出題する（番号の代わりに答えの文字列を直接入力してもよい）。Excel の「あなたの解答」には番号ではなく選択肢の文字列を記録する。出題と判定に問題の種類による分岐は書かず、`ChoiceQuestion` が `ask()` を**オーバーライド**するだけで実現している（ポリモーフィズム）。

## クラスの関係（3役割・5クラス）

```text
【問題】 Question ─┬─ TextQuestion      記述式（親の ask() をそのまま使う）
                   └─ ChoiceQuestion    選択式（ask() をオーバーライド）
【記録】 AnswerRecord                   1問ぶんの回答記録（Question を1つ持つ）
【進行】 QuizRunner                     Question の ask() / check() を呼ぶだけ。
                                        結果は AnswerRecord にして受け取る
```

## 実行方法

上から順にすべてのセルを実行する。`main` のセルで名前と解答を入力する。

### 1. Google Drive のマウント

`quiz_generator.ipynb`・`quiz.json`・結果の Excel はすべて Drive 上に置く。`/content/` 直下はセッション終了で消えてしまうため、最初に Drive をマウントする。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. quiz_generator.ipynb との連携

`%run` で quiz_generator.ipynb を実行し、Drive 上の quiz.json を介してデータを受け渡している。クイズの生成はこのセルが担当し、後で定義する `QuizRunner` は quiz.json を読むことだけに責任を持つ（生成と出題の分離）。

- quiz_generator.ipynb は読む・実行するだけで、編集も quiz.json の書き換えもしない
- `-i` オプションは付けない。ただし `.ipynb` を `%run` するとセルはこのノートブックの名前空間でそのまま実行されるため、`-i` の有無にかかわらず名前空間は共有される
- 実行後、generator の `quiz_data`・`save_path`・`f`・`drive` がグローバルに流れ込む。このノートブックでは `drive` をマウントにしか使わず、`quiz_data`・`save_path`・`f` は使わない。状態はすべてインスタンス変数に持たせる

In [ ]:
%run /content/drive/MyDrive/quiz_generator.ipynb

### 3. import

使うモジュールをまとめて読み込む。

| モジュール | 用途 |
| --- | --- |
| `json` | quiz.json の読み込み |
| `random` | 問題のランダム抽出（`random.sample`） |
| `unicodedata`, `re` | 解答の表記揺れの吸収（全角半角・空白の整理）、ファイル名の安全化 |
| `os` | 保存先パスの結合 |
| `datetime`, `ZoneInfo` | 日本時間（JST）のタイムスタンプ（Colab は UTC で動くため） |
| `pandas` | 結果を表にして Excel に出力 |

In [ ]:
import json
import random
import unicodedata
import re
import os
from datetime import datetime
from zoneinfo import ZoneInfo

import pandas as pd

### 4. Question 系クラス（問題）

使っている概念:

- **カプセル化**: データは `_question` のようにアンダースコア付きで持ち、`@property` で公開する。setter は作らないので `q.question = ...` は `AttributeError` になる（読み取り専用）。`_` 付きの直接代入は Python では防げないため「触らない約束」として使う
- **継承とオーバーライド**: `TextQuestion`（記述式）と `ChoiceQuestion`（選択式）が `Question` を継承する。`check` などは親のものを共用し、`ChoiceQuestion` だけが `ask()` を差し替える
- **ポリモーフィズム**: 呼び出す側は `q.ask()` / `q.check()` と書くだけで、記述式か選択式かを区別しない。なお `ask()` の入力プロンプトを「答えを入力: 」にしているのは、入力後に `QuizRunner` が表示する「あなたの解答: ...」と同じ行が二重に出ないようにするため
- **staticmethod / classmethod**: `normalize` は状態を使わないので `@staticmethod`、`from_dict` は dict から適切な子クラスを作る生成用の `@classmethod`。基底クラスが子クラスを知る点は妥協で、種類が増えたら `from_dict` だけを直せばよい

In [ ]:
class Question:
    """1問ぶんの問題を表す基底クラス。

    問題文・正解・カテゴリを読み取り専用で保持し、出題 (ask) と
    判定 (check) を担当する。記述式と選択式に共通する部分をここに置く。
    """

    def __init__(self, question, answer, category="不明"):
        """問題文・正解・カテゴリを受け取って保持する。"""
        self._question = question
        self._answer = answer
        self._category = category

    @property
    def question(self):
        """問題文（読み取り専用）。"""
        return self._question

    @property
    def answer(self):
        """正解（読み取り専用）。"""
        return self._answer

    @property
    def category(self):
        """カテゴリ（読み取り専用）。"""
        return self._category

    @staticmethod
    def normalize(text) -> str:
        """表記揺れを吸収した文字列を返す。

        全角半角を統一 (NFKC) し、前後の空白を除いて小文字にし、連続する
        空白を1つにまとめる。"while True" のように答えの中に空白を含む
        ものがあるため、空白は削除しない。
        """
        text = unicodedata.normalize("NFKC", str(text))
        text = text.strip().lower()
        return re.sub(r"\s+", " ", text)

    def check(self, user_answer) -> bool:
        """解答が正解と一致するかを、表記揺れを吸収して判定する。"""
        return self.normalize(user_answer) == self.normalize(self._answer)

    def ask(self) -> str:
        """問題文を表示して解答を入力させ、入力された文字列を返す。

        プロンプトを「あなたの解答」にしないのは、入力後に QuizRunner が
        「あなたの解答: ...」を表示するため（二重表示を避ける）。
        """
        print(f"問題文: {self._question}")
        return input("答えを入力: ")

    @classmethod
    def from_dict(cls, d):
        """quiz.json の1問ぶんの dict から、適切な問題オブジェクトを作る。

        choices があれば ChoiceQuestion、なければ TextQuestion を返す。
        問題を作るときの種類による分岐はここだけに集め、呼び出し側には書かせない。
        """
        category = d.get("category", "不明")
        choices = d.get("choices")
        if choices:
            return ChoiceQuestion(d["question"], d["answer"], category,
                                  choices)
        return TextQuestion(d["question"], d["answer"], category)


class TextQuestion(Question):
    """記述式の問題。

    親の ask() をそのまま使うので本体は空でよい。それでも別クラスに
    しているのは、記述式と選択式を Question の対等な兄弟クラスとして
    扱うため。
    """


class ChoiceQuestion(Question):
    """選択式の問題。

    選択肢を番号付きで表示し、番号でも答えの文字列でも解答できる。
    ask() をオーバーライドするだけで、判定 (check) は親のものを使う。
    """

    def __init__(self, question, answer, category="不明", choices=()):
        """親の初期化に加えて、選択肢をタプルで保持する。"""
        super().__init__(question, answer, category)
        self._choices = tuple(choices)
        if not self._choices:
            raise ValueError("選択肢が空です。")

    @property
    def choices(self):
        """選択肢のタプル（読み取り専用）。"""
        return self._choices

    def ask(self) -> str:
        """選択肢を番号付きで表示し、解答の文字列を返す。

        番号が入力されたら対応する選択肢の文字列を返す。番号以外は答えそのもの
        とみなして、そのまま返す。範囲外の番号は再入力させる。
        """
        print(f"問題文: {self.question}")
        for number, choice in enumerate(self._choices, start=1):
            print(f"  {number}. {choice}")
        while True:
            typed = input("番号または答えを入力: ")
            text = self.normalize(typed)
            # normalize で全角数字も半角になるので、数字だけなら番号とみなす
            if not text.isdecimal():
                return typed
            selected = int(text)
            if 1 <= selected <= len(self._choices):
                return self._choices[selected - 1]
            print(f"1〜{len(self._choices)} の番号を入力してください。")

### 5. AnswerRecord クラス（回答記録）

使っている概念:

- **カプセル化**: 1問ぶんの結果（問題番号・問題・解答・正誤）をすべて読み取り専用の `@property` で公開する
- **計算プロパティ**: `judgement`（○ / ×）は値を保存せず、`is_correct` から毎回計算する。保存しないので `is_correct` と食い違うことがない
- **to_row**: Excel の1行ぶんを dict で返す。キーの順番がそのまま Excel の列順になる

In [ ]:
class AnswerRecord:
    """1問ぶんの回答記録。

    出題した問題・入力された解答・正誤を、読み取り専用で保持する。
    """

    def __init__(self, number, question, user_answer, is_correct):
        """問題番号・Question・あなたの解答・正誤を受け取って保持する。"""
        self._number = number
        self._question = question
        self._user_answer = user_answer
        self._is_correct = is_correct

    @property
    def number(self):
        """問題番号（読み取り専用）。"""
        return self._number

    @property
    def question(self):
        """出題した Question オブジェクト（読み取り専用）。"""
        return self._question

    @property
    def user_answer(self):
        """あなたの解答（読み取り専用）。"""
        return self._user_answer

    @property
    def is_correct(self):
        """正解なら True（読み取り専用）。"""
        return self._is_correct

    @property
    def judgement(self):
        """判定の記号 "○" / "×"。is_correct から毎回計算する。"""
        return "○" if self._is_correct else "×"

    def to_row(self) -> dict:
        """Excel の1行ぶんを dict で返す。キーの順番が列の順番になる。"""
        return {
            "問題番号": self._number,
            "問題文": self._question.question,
            "あなたの解答": self._user_answer,
            "正解": self._question.answer,
            "判定": self.judgement,
        }

### 6. QuizRunner クラス（進行）

使っている概念:

- **クラス変数**: `DEFAULT_PATH`・`OUTPUT_DIR`・`NUM_QUESTIONS`・`TZ` を、全インスタンス共通の設定として持つ
- **プロパティ**: `player_name` だけ setter 付き（空の名前を `ValueError` で拒否し、ファイル名に使えない文字を `_` に置換）。`questions`・`results`・`score`・`accuracy` は読み取り専用で、リストは `tuple` にして返す（外から変更されない）
- **ポリモーフィズム**: 出題と判定では問題の種類を見ず、`q.ask()` と `q.check()` を呼ぶだけ。選択式を足しても `run_quiz` は変わらない（種類を数えるのは `load_quiz` の件数表示だけ）
- `__init__` はパスを保持するだけでファイルは読まない（読み込みは `load_quiz` の役目）

補足: 結果の Excel では、`=` で始まる解答（`==` など）が数式として書かれて壊れないよう、文字列セルとして保存する。また問題が1問も無いまま `run_quiz` が呼ばれても落ちないよう、案内を出して戻るガードを入れている。

In [ ]:
class QuizRunner:
    """クイズの読み込み・出題・集計・Excel 保存を行うクラス。

    出題と判定では問題の種類を見ず、Question の ask() と check() を
    呼ぶだけ。結果は AnswerRecord のリストとして持つ。
    """

    DEFAULT_PATH = "/content/drive/MyDrive/quiz.json"
    OUTPUT_DIR = "/content/drive/MyDrive"
    NUM_QUESTIONS = 5
    TZ = ZoneInfo("Asia/Tokyo")

    def __init__(self, quiz_file_path=DEFAULT_PATH):
        """クイズファイルのパスを保持し、状態を空で初期化する。

        ファイルはここでは読まない（読み込みは load_quiz で行う）。
        """
        self._quiz_file_path = quiz_file_path
        self._questions = []
        self._results = []
        self._player_name = "player"

    @property
    def player_name(self):
        """ファイル名に使える形に整えたプレイヤー名。"""
        return self._player_name

    @player_name.setter
    def player_name(self, value):
        """名前を設定する。空文字・空白のみは ValueError で拒否する。

        ファイル名に使えない文字と空白は "_" に置換して保存する。
        """
        if not isinstance(value, str) or not value.strip():
            raise ValueError("名前が空です。1文字以上入力してください。")
        self._player_name = re.sub(r'[\\/:*?"<>|\s]', "_", value.strip())

    @property
    def questions(self):
        """読み込んだ問題のタプル（読み取り専用）。"""
        return tuple(self._questions)

    @property
    def results(self):
        """回答記録 AnswerRecord のタプル（読み取り専用）。"""
        return tuple(self._results)

    @property
    def score(self):
        """正解数。回答記録から毎回計算する。"""
        return sum(1 for r in self._results if r.is_correct)

    @property
    def accuracy(self):
        """正答率 (0.0〜1.0)。結果が0件なら 0.0。"""
        if not self._results:
            return 0.0
        return self.score / len(self._results)

    def load_quiz(self):
        """クイズファイルを読み込み、Question のリストにして保持する。

        ファイルがなければ案内を表示したうえで、例外はそのまま再送出する。
        """
        try:
            with open(self._quiz_file_path, encoding="utf-8") as quiz_file:
                data = json.load(quiz_file)
        except FileNotFoundError:
            print(f"{self._quiz_file_path} が見つかりません。"
                  "先に quiz_generator.ipynb を実行してください。")
            raise
        self._questions = [Question.from_dict(d) for d in data["questions"]]
        # 件数表示のためだけに種類を数える（出題・判定では種類を見ない）
        choice_count = sum(isinstance(q, ChoiceQuestion)
                           for q in self._questions)
        print(f"{len(self._questions)} 問を読み込みました"
              f"（うち選択式 {choice_count} 問）")

    def run_quiz(self):
        """問題をランダムに NUM_QUESTIONS 問選んで出題し、結果を記録する。

        問題数が足りなければ全問を出題する。出題と判定は Question に任せる。
        """
        self._results = []
        if not self._questions:
            print("出題できる問題がありません。load_quiz() の実行と、"
                  "quiz.json の中身を確認してください。")
            return
        count = min(self.NUM_QUESTIONS, len(self._questions))
        chosen = random.sample(self._questions, count)
        for number, q in enumerate(chosen, start=1):
            print(f"\n【第{number}問】カテゴリ: {q.category}")
            user_answer = q.ask()
            record = AnswerRecord(number, q, user_answer, q.check(user_answer))
            self._results.append(record)
            print(f"あなたの解答: {record.user_answer}")
            print(f"判定: {record.judgement}")
            if not record.is_correct:
                print(f"正解: {q.answer}")

    def show_summary(self):
        """正解数と正答率を表示する。"""
        print(f"\n結果: {self.score} / {len(self._results)} 問正解"
              f"（正答率 {self.accuracy:.0%}）")

    def save_results_to_excel(self) -> str:
        """結果を Excel ファイルに保存し、保存先のパスを返す。

        結果が空なら保存せずに警告し、空文字を返す。ファイル名の日時は
        日本時間 (TZ) で作る。
        """
        if not self._results:
            print("保存する結果がありません。先に run_quiz() を実行して"
                  "ください。")
            return ""
        df = pd.DataFrame([r.to_row() for r in self._results])
        now = datetime.now(self.TZ)
        file_name = f"quiz_result_{self._player_name}_{now:%Y%m%d_%H%M%S}.xlsx"
        output_path = os.path.join(self.OUTPUT_DIR, file_name)
        with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
            df.to_excel(writer, sheet_name="Sheet1", index=False)
            # "==" のように "=" で始まる解答が数式として書かれて Excel が
            # 壊れないよう、文字列セルとして保存する
            for row in writer.sheets["Sheet1"].iter_rows():
                for cell in row:
                    text = cell.value
                    if isinstance(text, str) and text.startswith("="):
                        cell.data_type = "s"
        print(f"結果を保存しました: {output_path}")
        return output_path

### 7. main

実行の流れを `main()` にまとめる。名前入力 → `load_quiz` → `run_quiz` → `show_summary` → `save_results_to_excel` の順。名前が空のときは `ValueError` を受けて再入力を促す。`if __name__ == "__main__":` を付け、import されただけでは動かないようにする。

`main` 内のローカル変数は、generator から流れ込む `quiz_data`・`save_path`・`f`・`drive` と同名にしない。

In [ ]:
def main():
    """名前入力から Excel 保存までのクイズ全体の流れを実行する。"""
    runner = QuizRunner()
    while True:
        try:
            runner.player_name = input("名前（アルファベット）を入力: ")
            break
        except ValueError as error:
            print(f"入力エラー: {error}")
    runner.load_quiz()
    runner.run_quiz()
    runner.show_summary()
    runner.save_results_to_excel()


if __name__ == "__main__":
    main()

### 動作確認（from_dict と判定：選択式・記述式）

5問のランダム抽出では選択式が出ないこともあるため、`from_dict` で問題を直接作って確認する。入力を伴う `ask()` は呼ばない。判定は `normalize` を通すので、全角・大文字・前後の空白があっても正解になる。

In [ ]:
choice_sample = Question.from_dict({
    "question": "Pythonで関数を定義するキーワードは何ですか？",
    "answer": "def",
    "category": "関数",
    "choices": ["func", "def", "define", "lambda"],
})
print(type(choice_sample))
print(choice_sample.choices)
print('check("def")        ->', choice_sample.check("def"))
print('check(" ＤＥＦ ")   ->', choice_sample.check(" ＤＥＦ "))
print('check("func")       ->', choice_sample.check("func"))

text_sample = Question.from_dict({"question": "繰り返しの構文は？",
                                  "answer": "for"})
print(type(text_sample), text_sample.category)
print('check("　ｆｏｒ ")   ->', text_sample.check("　ｆｏｒ "))